# ML Feature Engineering for Equities (demo)
- Lag returns, rolling volatility, RSI
- Train/test split for a classifier (direction)


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier

sym = 'AAPL'
df = yf.download(sym, start='2016-01-01', progress=False)
df = df[['Adj Close']].rename(columns={'Adj Close':'px'})
df['ret1'] = df['px'].pct_change()
df['ret5'] = df['px'].pct_change(5)
df['vol20'] = df['ret1'].rolling(20).std()
df['rsi14'] = 100 - 100/(1 + df['ret1'].clip(lower=0).rolling(14).mean() / df['ret1'].clip(upper=0).abs().rolling(14).mean())
df['target'] = (df['ret1'].shift(-1) > 0).astype(int)
df = df.dropna()

X = df[['ret1','ret5','vol20','rsi14']]
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
model = LGBMClassifier(max_depth=3, n_estimators=200, learning_rate=0.05)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred))
